# OTP-FM Quick Start: Gaussian Experiments

This notebook demonstrates OTP-FM (Optimal Transport Potentials for Multi-Marginal Flow Matching) on simple 1D and 2D Gaussian distributions.

In [ ]:
import torch
import numpy as np
from collections import OrderedDict

# Import OTP-FM
from otpfm import OTPFM
from otpfm.potentials import IndependentPotential, W2Potential
from otpfm.lambda_functions import GaussianLambda

## 1. Define Marginal Distributions

We'll create three Gaussian distributions:
- Source: N(0, 1)
- Intermediate at t=0.5: N(2, 0.5)
- Target: N(4, 1)

In [ ]:
# Sample from marginals
n_samples = 1000
dim = 1

x0 = torch.randn(n_samples, dim) * 1.0 + 0.0  # Source
x_half = torch.randn(n_samples, dim) * 0.5 + 2.0  # Intermediate
x1 = torch.randn(n_samples, dim) * 1.0 + 4.0  # Target

print(f"Source mean: {x0.mean():.2f}, std: {x0.std():.2f}")
print(f"Intermediate mean: {x_half.mean():.2f}, std: {x_half.std():.2f}")
print(f"Target mean: {x1.mean():.2f}, std: {x1.std():.2f}")

## 2. Create OTP-FM Model

We define a potential at t=0.5 to enforce the intermediate marginal constraint.

In [ ]:
# Define potential at intermediate time
potential = IndependentPotential(
    tk=0.5,
    strength=100.0,
    lambda_fn_type='gaussian',
    width=0.1
)

potentials = OrderedDict({0.5: potential})

# Create model
model = OTPFM(
    d=dim,
    tks=[0.5],
    potentials=potentials,
    flownet_args={
        'hidden_dim': 128,
        'num_hidden_layers': 2,
    }
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Training

Train the model using the MeanFlow consistency loss with OT potential corrections.

In [ ]:
# Prepare training data: (batch_size, num_marginals, dim)
# Stack [x0, x_half, x1]
batch_size = 64
n_epochs = 100

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(n_epochs):
    model.train()
    
    # Sample batch
    idx = torch.randperm(n_samples)[:batch_size]
    xs = torch.stack([x0[idx], x_half[idx], x1[idx]], dim=1)  # (batch, 3, dim)
    
    # Progressive OTP alpha (sigmoid schedule)
    otp_alpha = 1.0 / (1.0 + np.exp(-6.0 * (epoch - n_epochs/2) / (n_epochs/2)))
    
    # Forward pass
    loss = model.forward_with_losses(xs, otp_alpha, do_otp=True)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update EMA
    model.update_ema()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}, otp_alpha={otp_alpha:.2f}")

## 4. Sample Trajectories

Generate trajectories from the trained model.

In [ ]:
# Sample trajectories
model.eval()

n_test = 100
test_x0 = torch.randn(n_test, dim)

with torch.no_grad():
    trajectories, t_eval = model.sample(test_x0, n_steps=20, ema=True)

print(f"Trajectory shape: {trajectories.shape}")
print(f"Time points: {t_eval.shape}")

## 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

# Plot trajectories
for i in range(min(50, n_test)):
    ax.plot(t_eval.numpy(), trajectories[:, i, 0].numpy(), 
            alpha=0.3, color='purple', lw=0.5)

# Plot marginal means
ax.axhline(0.0, color='red', ls='--', label='Source mean')
ax.axhline(2.0, color='orange', ls='--', label='Intermediate mean (t=0.5)')
ax.axhline(4.0, color='green', ls='--', label='Target mean')

ax.axvline(0.5, color='gray', ls=':', alpha=0.5)

ax.set_xlabel('Time t')
ax.set_ylabel('x(t)')
ax.set_title('OTP-FM Trajectories')
ax.legend()
plt.tight_layout()
plt.show()

## Next Steps

- Try different potential types: `W2Potential`, `EntropicW2Potential`, `KLPotential`
- Experiment with different lambda functions: `delta`, `triangle`, `box`
- See other notebooks for real-world applications:
  - `02_singlecell_eb.ipynb`: Single-cell trajectory inference
  - `03_gulf_of_mexico.ipynb`: Ocean current modeling
  - `04_beijing_airquality.ipynb`: Air quality forecasting